## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([28*28, 100]))   # 输入层到隐藏层的权重
        self.b1 = tf.Variable(tf.random.normal([100]))          # 隐藏层的偏置
        self.W2 = tf.Variable(tf.random.normal([100, 10]))      # 隐藏层到输出层的权重
        self.b2 = tf.Variable(tf.random.normal([10]))           # 输出层的偏置

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, (-1, 28*28))                      # 将输入展平为 [batch_size, 28*28]
        h1 = tf.matmul(x, self.W1) + self.b1                # 第一层线性变换
        h1_relu = tf.nn.relu(h1)                            # ReLU激活
        logits = tf.matmul(h1_relu, self.W2) + self.b2      # 第二层线性变换
        
        return logits

        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [9]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 11.389548 ; accuracy 0.59991664
epoch 1 : loss 11.357616 ; accuracy 0.60073334
epoch 2 : loss 11.3259535 ; accuracy 0.6013833
epoch 3 : loss 11.294548 ; accuracy 0.60215
epoch 4 : loss 11.26339 ; accuracy 0.603
epoch 5 : loss 11.232485 ; accuracy 0.60373336
epoch 6 : loss 11.201836 ; accuracy 0.6046
epoch 7 : loss 11.171432 ; accuracy 0.6052833
epoch 8 : loss 11.141261 ; accuracy 0.6059833
epoch 9 : loss 11.111335 ; accuracy 0.6066
epoch 10 : loss 11.081639 ; accuracy 0.6073
epoch 11 : loss 11.052171 ; accuracy 0.6079
epoch 12 : loss 11.022927 ; accuracy 0.60868335
epoch 13 : loss 10.993905 ; accuracy 0.6095667
epoch 14 : loss 10.965102 ; accuracy 0.61011666
epoch 15 : loss 10.936517 ; accuracy 0.61088336
epoch 16 : loss 10.908148 ; accuracy 0.6116833
epoch 17 : loss 10.879998 ; accuracy 0.61226666
epoch 18 : loss 10.852062 ; accuracy 0.61305
epoch 19 : loss 10.824346 ; accuracy 0.6138
epoch 20 : loss 10.796831 ; accuracy 0.61448336
epoch 21 : loss 10.769525 ; accuracy 0